# 09 — Irish Web Inference

Process the full Irish web through the fine-tuned model using RunPod Serverless.
Target: ~128,000 Irish SME sites (~640,000 pages).

**Infrastructure**: RunPod Serverless with auto-scaling to 4× L40S workers.
- Cost: ~$100-250 for full run
- Duration: ~2.5 days at 4× L40S
- Scale-to-zero when idle

**Output**: `data/results/irish_web_results.jsonl` — one JSON per site with:
- existing_schema (from WDC, if any)
- generated_schema (model output)
- improvement_delta
- quality_score

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
import asyncio
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

RESULTS_DIR = Path('../data/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

from src.runpod_utils import init_runpod, get_endpoint_status, estimate_serverless_cost
from src.schema_validator import validate_jsonld, extract_properties

ENDPOINT_ID = os.getenv('RUNPOD_ENDPOINT_ID')
init_runpod()

print(f'Endpoint ID: {ENDPOINT_ID}')
if ENDPOINT_ID:
    status = get_endpoint_status(ENDPOINT_ID)
    print(f'Endpoint status: {status}')

## Load Irish Web Target Sites

In [ ]:
# Load .ie domain list from notebook 01
cc_domains = []
cc_path = Path('../data/raw/ie_domains_cc_api.jsonl')
if cc_path.exists():
    with open(cc_path) as f:
        for line in f:
            cc_domains.append(json.loads(line))

# Load existing schema data from WDC
wdc_by_domain = {}
wdc_path = Path('../data/raw/wdc_ie_records.jsonl')
if wdc_path.exists():
    from urllib.parse import urlparse
    with open(wdc_path) as f:
        for line in f:
            rec = json.loads(line)
            domain = urlparse(rec.get('source_url', '')).netloc.lstrip('www.')
            wdc_by_domain[domain] = rec

print(f'Target .ie domains: {len(cc_domains)}')
print(f'Domains with existing WDC schema: {len(wdc_by_domain)}')
print(f'Domains without schema: {len(cc_domains) - len(wdc_by_domain)}')

In [ ]:
# Cost estimate before starting
n_pages = len(cc_domains)
for workers in [1, 2, 4]:
    est = estimate_serverless_cost(n_pages, avg_seconds_per_page=1.44)
    print(f'Estimate ({n_pages:,} pages): ${est["estimated_cost_usd"]:.0f}, {est["estimated_hours"]:.0f}h on 1×L40S')
    print(f'  With 4 workers: ~${est["estimated_cost_usd"]/1:.0f}, ~{est["estimated_hours"]/4:.0f}h')
    break

## Build Processing Queue

In [ ]:
from tqdm.auto import tqdm

HTML_DIR = Path('../data/raw/html')
SCREENSHOT_DIR = Path('../data/screenshots')

done_domains = set()
results_path = RESULTS_DIR / 'irish_web_results.jsonl'
if results_path.exists():
    with open(results_path) as f:
        for line in f:
            rec = json.loads(line)
            done_domains.add(rec.get('domain', ''))

print(f'Already processed: {len(done_domains)} domains')
print(f'Building queue from {len(cc_domains)} .ie domains...')

queue = []
for domain_rec in tqdm(cc_domains, desc="Building queue"):
    domain = domain_rec.get('_domain', '')
    if domain in done_domains:
        continue

    url = domain_rec.get('sample_url') or domain_rec.get('url', '')
    html_path = HTML_DIR / f"{url.replace('://', '_').replace('/', '_')[:100]}.html"

    if not html_path.exists():
        continue

    screenshot_path = SCREENSHOT_DIR / f'{domain}.png'
    queue.append({
        'id': domain,
        'url': url,
        'html_path': str(html_path),
        'screenshot_path': str(screenshot_path) if screenshot_path.exists() else None,
        'existing_schema': wdc_by_domain.get(domain, {}).get('jsonld'),
    })

print(f'Items in queue: {len(queue)}')

## Run Batch Inference

In [ ]:
from src.inference import RunPodSchemaClient

RUNPOD_API_KEY = os.getenv('RUNPOD_API_KEY')

async def process_batch_and_save(
    items: list[dict],
    output_path: str,
    batch_size: int = 100,
    concurrency: int = 8,
):
    client = RunPodSchemaClient(endpoint_id=ENDPOINT_ID, api_key=RUNPOD_API_KEY)
    
    total = len(items)
    processed = 0
    
    with open(output_path, 'a') as out_file:
        for i in range(0, total, batch_size):
            batch = items[i:i + batch_size]
            
            # Load HTML for batch
            for item in batch:
                with open(item['html_path'], 'r', errors='replace') as f:
                    item['html'] = f.read()
            
            results = await client.process_batch_async(batch, concurrency=concurrency)
            
            for item, result in zip(batch, results):
                if not result.get('success'):
                    continue
                
                schema_str = result.get('schema_jsonld', '')
                validation = validate_jsonld(schema_str) if schema_str else {'valid': False, 'quality_score': 0}
                
                # Compute improvement delta vs existing schema
                existing = item.get('existing_schema')
                improvement = {}
                if existing:
                    existing_str = json.dumps(existing)
                    existing_val = validate_jsonld(existing_str)
                    existing_props = set(existing.keys()) - {'@context', '@type', '@id'}
                    gen_props = set(json.loads(schema_str).keys()) - {'@context', '@type', '@id'} if validation['valid'] else set()
                    improvement = {
                        'new_properties': list(gen_props - existing_props),
                        'existing_property_count': len(existing_props),
                        'generated_property_count': len(gen_props),
                        'quality_delta': round(validation['quality_score'] - existing_val['quality_score'], 3),
                    }
                
                output_rec = {
                    'domain': item['id'],
                    'url': item['url'],
                    'existing_schema': existing,
                    'generated_schema': json.loads(schema_str) if validation['valid'] else None,
                    'schema_types': validation.get('schema_types', []),
                    'quality_score': validation['quality_score'],
                    'properties_count': validation.get('property_count', 0),
                    'improvement_delta': improvement,
                }
                out_file.write(json.dumps(output_rec, ensure_ascii=False) + '\n')
            
            processed += len(batch)
            print(f'Progress: {processed}/{total} ({100*processed/total:.1f}%)')

print('Processing function defined')

RUN_INFERENCE = False  # Set True to run full Irish web inference
if RUN_INFERENCE and ENDPOINT_ID and queue:
    await process_batch_and_save(queue, str(results_path), batch_size=100, concurrency=8)

In [ ]:
# Check results so far
if results_path.exists():
    results = []
    with open(results_path) as f:
        for line in f:
            results.append(json.loads(line))
    
    print(f'Results so far: {len(results)}')
    
    valid = [r for r in results if r.get('generated_schema')]
    with_existing = [r for r in results if r.get('existing_schema')]
    print(f'Valid generated schema: {len(valid)} ({100*len(valid)/len(results):.1f}%)')
    print(f'Had existing schema: {len(with_existing)} ({100*len(with_existing)/len(results):.1f}%)')
    
    if valid:
        avg_quality = sum(r['quality_score'] for r in valid) / len(valid)
        avg_props = sum(r['properties_count'] for r in valid) / len(valid)
        print(f'Avg quality score: {avg_quality:.3f}')
        print(f'Avg property count: {avg_props:.1f}')
else:
    print('No results yet — set RUN_INFERENCE=True above to start processing')

print('\nNext step: 10_analysis_dashboard.ipynb')